In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:

# --- Install ---
!pip -q install open_clip_torch torch torchvision pandas tqdm faiss-cpu
!pip install -U bitsandbytes>=0.46.1


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 70.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 75.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.8 MB/s eta 0:00:00


In [ ]:
import os, time, random
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn.functional as F
import open_clip

# -----------------------------
# Config
# -----------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

FRAMES_ROOT = "/content/drive/MyDrive/MSR_VTT/frames"     # TODO
CAPTIONS_CSV = "/content/drive/MyDrive/MSR_VTT/index.csv" # TODO, columns: video_id, caption
VEDIO_EMBEDD = "/content/drive/MyDrive/MSR_VTT/msrvtt_video_embs_shahd.npy"
NARRATION_PATH = "/content/drive/MyDrive/MSR_VTT/video_narrations_blip2.json"


MODEL_NAME = "ViT-B-32"
PRETRAINED = "openai"

NUM_FRAMES = 12          # sample N frames per video
BATCH_TEXT = 256
BATCH_IMAGE = 64         # image batch for frame encoding (memory-safe)

from pathlib import Path

# -------- Google Drive ROOT --------
DRIVE_ROOT = Path("/content/drive/MyDrive/MSR_VTT")


# -------- RAW DATA --------
VIDEOS_DIR = DRIVE_ROOT / "raw/TrainValVideo"
CAPTIONS_DIR = DRIVE_ROOT / "raw/captions"

# -------- OUTPUT --------


OUT_PROCESSED_DIR = FRAMES_ROOT
INDEX_PARQUET = DRIVE_ROOT / "index.parquet"
INDEX_CSV = DRIVE_ROOT / "index.csv"
INDEX_PARQUET.parent.mkdir(parents=True, exist_ok=True)

print(DEVICE)

print("Parquet exists:", INDEX_PARQUET.exists())
print("CSV exists:", INDEX_CSV.exists())

cuda
Parquet exists: True
CSV exists: True


In [ ]:
print(f"Number of files in {VIDEOS_DIR}: {len(list(VIDEOS_DIR.glob('*')))}")

Number of files in /content/drive/MyDrive/MSR_VTT/raw/TrainValVideo: 7010


In [ ]:
import re

# Get all video files in the directory
existing_video_paths = list(VIDEOS_DIR.glob("*.mp4"))

# Extract video numbers from filenames (e.g., video0.mp4 -> 0)
existing_video_ids = set()
for path in existing_video_paths:
    match = re.match(r"video(\d+)\.mp4", path.name)
    if match:
        existing_video_ids.add(int(match.group(1)))

# Define the expected range of video IDs
expected_video_ids = set(range(0, 7010)) # 0 to 7009 inclusive

# Find missing video IDs
missing_ids = sorted(list(expected_video_ids - existing_video_ids))

print(f"Expected video count (0-7009): {len(expected_video_ids)}")
print(f"Found video count: {len(existing_video_ids)}")

if missing_ids:
    print(f"\nFound {len(missing_ids)} missing videos:")
    for x in missing_ids:
        print(x)
else:
    print("\nNo missing videos found in the range 0-7009.")

Expected video count (0-7009): 7010
Found video count: 7010

No missing videos found in the range 0-7009.


## Frames Extraction

In [ ]:
import os, json, re, subprocess
from collections import defaultdict
import pandas as pd
from pathlib import Path
from tqdm import tqdm


FPS = 8.0  # baseline (already applied)

CAPTION_JSON = CAPTIONS_DIR / "caption.json"
INFO_JSON    = CAPTIONS_DIR / "info.json"

# ============================================================
# 🚀 FAST EXIT: dataset already prepared
# ============================================================
frames_root = Path(FRAMES_ROOT)

if INDEX_PARQUET.exists() and frames_root.exists() and any(frames_root.iterdir()):
    print("✅ Dataset already prepared.")
    print(" - Frames directory exists and is non-empty")
    print(" - Index file exists:", INDEX_PARQUET)
    print("⏭️ Skipping extraction and indexing to avoid wasted time.")
else:
    # ---------------------------
    # Helpers
    # ---------------------------
    def normalize_video_key(x: str) -> str:
        s = str(x).strip().replace("\\", "/").split("/")[-1]
        s = re.sub(r"\.mp4$", "", s)
        if re.fullmatch(r"\d+", s):
            s = f"video{s}"
        return s

    def load_json(p):
        with open(p, "r", encoding="utf-8") as f:
            return json.load(f)

    def parse_captions(caption_data):
        caps = defaultdict(list)

        if isinstance(caption_data, list):
            for it in caption_data:
                if not isinstance(it, dict):
                    continue
                vid = it.get("video_id") or it.get("video") or it.get("clip_name") or it.get("vid") or it.get("id")
                cap = it.get("caption") or it.get("sentence") or it.get("text") or it.get("description")
                if vid and cap:
                    caps[normalize_video_key(vid)].append(str(cap).strip())
            return dict(caps)

        if not isinstance(caption_data, dict):
            raise ValueError("caption.json must be dict or list")

        for k, v in caption_data.items():
            vid = normalize_video_key(k)
            if isinstance(v, str):
                caps[vid].append(v.strip())
            elif isinstance(v, list):
                for item in v:
                    if isinstance(item, str):
                        caps[vid].append(item.strip())
            elif isinstance(v, dict):
                for key in ("captions", "sentences", "annotations", "caption", "sentence", "text", "description"):
                    c = v.get(key)
                    if isinstance(c, str):
                        caps[vid].append(c.strip())
                    elif isinstance(c, list):
                        for item in c:
                            if isinstance(item, str):
                                caps[vid].append(item.strip())
        return dict(caps)

    def ffprobe_meta(video_path):
        cmd = [
            "ffprobe", "-v", "error",
            "-select_streams", "v:0",
            "-show_entries", "stream=width,height",
            "-show_entries", "format=duration",
            "-of", "json",
            str(video_path)
        ]
        p = subprocess.run(cmd, capture_output=True, text=True)
        if p.returncode != 0:
            return {}
        try:
            data = json.loads(p.stdout)
            return {
                "width": data["streams"][0].get("width"),
                "height": data["streams"][0].get("height"),
                "duration": float(data["format"].get("duration")),
            }
        except:
            return {}

    def extract_frames_ffmpeg(video_path, frames_dir, fps):
        frames_dir.mkdir(parents=True, exist_ok=True)
        out_pattern = str(frames_dir / "image_%05d.jpg")
        cmd = [
            "ffmpeg", "-hide_banner", "-loglevel", "error",
            "-i", str(video_path),
            "-vf", f"fps={fps}",
            out_pattern
        ]
        subprocess.run(cmd, check=False)

    def count_frames(frames_dir):
        return len(list(frames_dir.glob("*.jpg")))

    # ---------------------------
    # Load captions
    # ---------------------------
    caption_data = load_json(CAPTION_JSON)
    caps_by_video = parse_captions(caption_data)

    print("Loaded captions for videos:", len(caps_by_video))

    # ---------------------------
    # Process videos (ONE TIME ONLY)
    # ---------------------------
    video_paths = sorted(VIDEOS_DIR.glob("*.mp4"))
    assert video_paths, f"No .mp4 files found in {VIDEOS_DIR}"

    rows = []

    for vp in tqdm(video_paths, desc="Preparing dataset (one-time)"):
        vid = normalize_video_key(vp.stem)
        frames_dir = frames_root / vid
        frames_dir.mkdir(parents=True, exist_ok=True)

        if count_frames(frames_dir) == 0:
            extract_frames_ffmpeg(vp, frames_dir, FPS)

        caps = caps_by_video.get(vid, [])

        meta = {
            "video_id": vid,
            "frames_dir": str(frames_dir),
            "num_frames": count_frames(frames_dir),
        }
        meta.update(ffprobe_meta(vp))

        with open(frames_dir / "captions.json", "w", encoding="utf-8") as f:
            json.dump({"video_id": vid, "captions": caps}, f, indent=2)

        with open(frames_dir / "meta.json", "w", encoding="utf-8") as f:
            json.dump(meta, f, indent=2)

        for cap in caps:
            rows.append({
                "video_id": vid,
                "caption": cap,
                "frames_dir": str(frames_dir),
                "num_frames": meta["num_frames"],
                "duration": meta.get("duration"),
                "width": meta.get("width"),
                "height": meta.get("height"),
            })

    df = pd.DataFrame(rows)
    df.to_parquet(INDEX_PARQUET, index=False)
    df.to_csv(INDEX_CSV, index=False)

    print("✅ Dataset preparation completed and cached.")


✅ Dataset already prepared.
 - Frames directory exists and is non-empty
 - Index file exists: /content/drive/MyDrive/MSR_VTT/index.parquet
⏭️ Skipping extraction and indexing to avoid wasted time.


In [ ]:
print("Parquet exists:", INDEX_PARQUET.exists())
print("CSV exists:", INDEX_CSV.exists())

Parquet exists: True
CSV exists: True


In [ ]:
# -----------------------------
# Load caption–video index (prepared dataset)
# -----------------------------
import pandas as pd

# Use the prepared index, NOT raw captions
df_index = pd.read_csv(INDEX_CSV)

# Sanity check
assert {"video_id", "caption", "frames_dir"}.issubset(df_index.columns), \
    "❌ INDEX_CSV must contain video_id, caption, frames_dir columns"

# Unique videos to index
video_ids = sorted(df_index["video_id"].unique().tolist())

print("✅ Index loaded")
print("Unique videos:", len(video_ids))
print("Total query captions:", len(df_index))

# Optional preview
df_index.head()


✅ Index loaded
Unique videos: 7010
Total query captions: 140200


,video_id,caption,video_path,frames_dir,num_frames,duration,width,height
0,video0,a car is shown,/content/drive/MyDrive/MSR_VTT/raw/TrainValVid...,/content/drive/MyDrive/MSR_VTT/frames/video0,96,12.0,320,240
1,video0,a group is dancing,/content/drive/MyDrive/MSR_VTT/raw/TrainValVid...,/content/drive/MyDrive/MSR_VTT/frames/video0,96,12.0,320,240
2,video0,a man drives a vehicle through the countryside,/content/drive/MyDrive/MSR_VTT/raw/TrainValVid...,/content/drive/MyDrive/MSR_VTT/frames/video0,96,12.0,320,240
3,video0,a man drives down the road in an audi,/content/drive/MyDrive/MSR_VTT/raw/TrainValVid...,/content/drive/MyDrive/MSR_VTT/frames/video0,96,12.0,320,240
4,video0,a man driving a car,/content/drive/MyDrive/MSR_VTT/raw/TrainValVid...,/content/drive/MyDrive/MSR_VTT/frames/video0,96,12.0,320,240


In [ ]:
from pathlib import Path

FRAMES_ROOT = Path("/content/drive/MyDrive/MSR_VTT/frames")
NARRATION_PATH = Path("/content/drive/MyDrive/MSR_VTT/video_narrations_blip2.json")

print("FRAMES_ROOT =", FRAMES_ROOT)
print("Narration path =", NARRATION_PATH)
print("Exists?", NARRATION_PATH.exists())



FRAMES_ROOT = /content/drive/MyDrive/MSR_VTT/frames
Narration path = /content/drive/MyDrive/MSR_VTT/video_narrations_blip2.json
Exists? True


In [ ]:
# -----------------------------
# Narration generation (BLIP-2) — SAFE & SKIP IF EXISTS
# -----------------------------
import json
from pathlib import Path

#NARRATION_PATH = Path(FRAMES_ROOT).parent / "video_narrations_blip2.json"
NARRATION_PATH = Path("/content/drive/MyDrive/MSR_VTT/video_narrations_blip2.json")

# ✅ If narrations already exist, DO NOTHING
if NARRATION_PATH.exists():
    narrations = json.load(open(NARRATION_PATH, "r"))
    print("✅ Narrations already exist — skipping generation")
    print("Total narrations:", len(narrations))

else:
    print("🔄 Narrations not found — generating with BLIP-2")

    import torch
    from tqdm import tqdm
    from PIL import Image
    from transformers import AutoProcessor, Blip2ForConditionalGeneration

    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    MODEL_ID = "Salesforce/blip2-opt-2.7b"

    processor = AutoProcessor.from_pretrained(MODEL_ID)
    model = Blip2ForConditionalGeneration.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
        device_map="auto"
    )
    model.eval()

    narrations = {}

    def pick_middle_frame(video_id):
        frames = sorted((Path(FRAMES_ROOT) / video_id).glob("*.jpg"))
        return frames[len(frames)//2] if frames else None

    for vid in tqdm(video_ids, desc="Generating narrations"):
        frame = pick_middle_frame(vid)
        if frame is None:
            narrations[vid] = ""
            continue

        image = Image.open(frame).convert("RGB")

        inputs = processor(
            images=image,
            return_tensors="pt"
        ).to(model.device, torch.float16 if DEVICE == "cuda" else torch.float32)

        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=40
            )

        caption = processor.decode(out[0], skip_special_tokens=True)
        narrations[vid] = caption.strip()

        # crash-safe incremental save
        if len(narrations) % 50 == 0:
            json.dump(narrations, open(NARRATION_PATH, "w"), indent=2)

    json.dump(narrations, open(NARRATION_PATH, "w"), indent=2)
    print("✅ Narrations generated and saved:", NARRATION_PATH)


✅ Narrations already exist — skipping generation
Total narrations: 7010


# Video Encoding

Encodein video frames using CLIP with ViT-B-32 as backbone.

In [ ]:
# -----------------------------
# Load CLIP (CLIP4Clip-style baseline)
# -----------------------------

# 🔧 Backbone choice (paper default = ViT-B/16)
MODEL_NAME = "ViT-B-16"   # primary setting (paper)
# MODEL_NAME = "ViT-B-32" # optional analysis setting

PRETRAINED = "openai"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Query-aware filtering threshold (paper)
P_THRESH = 0.5 if MODEL_NAME == "ViT-B-16" else 0.4

model, _, preprocess = open_clip.create_model_and_transforms(
    MODEL_NAME,
    pretrained=PRETRAINED
)

tokenizer = open_clip.get_tokenizer(MODEL_NAME)
model = model.to(DEVICE).eval()

print(f"✅ Loaded CLIP backbone: {MODEL_NAME}")
print(f"   Query-aware threshold p = {P_THRESH}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


open_clip_model.safetensors:   0%|          | 0.00/599M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


✅ Loaded CLIP backbone: ViT-B-16
   Query-aware threshold p = 0.5


# **TRAINING**

In [ ]:
# -----------------------------
# Trainable text encoding (USED ONLY DURING TRAINING)
# -----------------------------
def encode_texts_trainable(texts):
    """
    Trainable text encoder for CLIP fine-tuning.
    IMPORTANT:
    - model MUST be in train() mode before calling this
    - do NOT call model.eval() here
    """
    toks = tokenizer(texts).to(DEVICE)
    text_emb = model.encode_text(toks)
    text_emb = F.normalize(text_emb, dim=-1)
    return text_emb


In [ ]:
# Reload captions dataframe (REQUIRED after kernel reset)
df = pd.read_csv(INDEX_CSV)
print("✅ df loaded:", df.shape)


✅ df loaded: (140200, 8)


In [ ]:
import os, glob
from pathlib import Path

ROOT = Path("/content/drive/MyDrive")
print("Drive mounted?", ROOT.exists())

# Search for the files anywhere under MyDrive
targets = [
    "msrvtt_video_embs_shahd.npy",
    "msrvtt_video_embs_shahd.ids.json",
    "msrvtt_video_embs.npy",
    "msrvtt_video_embs.json",
]

for t in targets:
    hits = list(ROOT.rglob(t))
    print(f"\n{t} -> {len(hits)} match(es)")
    for h in hits[:5]:
        print("  ", h)


Drive mounted? True

msrvtt_video_embs_shahd.npy -> 0 match(es)

msrvtt_video_embs_shahd.ids.json -> 0 match(es)

msrvtt_video_embs.npy -> 0 match(es)

msrvtt_video_embs.json -> 0 match(es)


In [ ]:
# -----------------------------
# Reload cached VIDEO embeddings (required for training)
# -----------------------------
import numpy as np
import json
from pathlib import Path

VIDEO_EMB_PATH = Path(VEDIO_EMBEDD)
IDS_PATH = VIDEO_EMB_PATH.with_suffix(".ids.json")

assert VIDEO_EMB_PATH.exists(), " video embeddings file missing"
assert IDS_PATH.exists(), " video ID file missing"

video_embs = np.load(VIDEO_EMB_PATH)
valid_video_ids = json.load(open(IDS_PATH))

vid2idx = {v: i for i, v in enumerate(valid_video_ids)}

print("✅ Reloaded cached video embeddings")
print("Embeddings shape:", video_embs.shape)
print("Indexed videos:", len(valid_video_ids))


✅ Reloaded cached video embeddings
Embeddings shape: (7010, 512)
Indexed videos: 7010


In [ ]:
# -----------------------------
# FAST OpenCLIP fine-tuning (TEXT ONLY, RESUMABLE)
# -----------------------------
import torch
import torch.nn.functional as F
from torch.optim import AdamW
from tqdm import tqdm
import random
import numpy as np
import os

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# -----------------------------
# SAFETY: reload df if kernel reset
# -----------------------------
df = pd.read_csv(INDEX_CSV)
print("✅ df loaded:", df.shape)

# -----------------------------
# Freeze vision, enable text
# -----------------------------
model.train()

# ❄️ Freeze vision encoder
for p in model.visual.parameters():
    p.requires_grad = False

# ✅ Train text transformer
for p in model.transformer.parameters():
    p.requires_grad = True

# ✅ Train text projection (if exists)
if hasattr(model, "text_projection") and model.text_projection is not None:
    model.text_projection.requires_grad = True

print("✅ Vision frozen, text trainable")

# -----------------------------
# Optimizer (TEXT ONLY)
# -----------------------------
optimizer = AdamW(
    list(model.transformer.parameters()) +
    ([model.text_projection] if model.text_projection is not None else []),
    lr=1e-5,
    weight_decay=1e-4
)

temperature = 0.07
EPOCHS = 2
BATCH_SIZE = 64

# -----------------------------
# Training pairs
# -----------------------------
train_pairs = list(zip(df["video_id"], df["caption"]))
random.shuffle(train_pairs)

print("Training pairs:", len(train_pairs))
print("Using cached video embeddings:", video_embs.shape)

# -----------------------------
# Checkpoint setup
# -----------------------------
CHECKPOINT_PATH = "openclip_text_finetuned_checkpoint.pt"

if os.path.exists(CHECKPOINT_PATH):
    ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    start_epoch = ckpt["epoch"] + 1
    print(f"🔁 Resumed from checkpoint at epoch {start_epoch}")
else:
    start_epoch = 0

# -----------------------------
# Training loop (SAFE & FAST)
# -----------------------------
for epoch in range(start_epoch, EPOCHS):
    random.shuffle(train_pairs)
    pbar = tqdm(
        range(0, len(train_pairs), BATCH_SIZE),
        desc=f"Epoch {epoch+1}/{EPOCHS}"
    )

    for i in pbar:
        batch = train_pairs[i:i + BATCH_SIZE]
        vids, texts = zip(*batch)

        # ---- Load cached VIDEO embeddings (NO re-encoding) ----
        video_feats = torch.from_numpy(
            np.stack([video_embs[vid2idx[v]] for v in vids])
        ).to(DEVICE)

        video_feats = F.normalize(video_feats, dim=-1)

        # ---- Encode TEXT (TRAINABLE path) ----
        text_feats = encode_texts_trainable(list(texts))



        # ---- CLIP contrastive loss ----
        logits = video_feats @ text_feats.T / temperature
        labels = torch.arange(len(batch), device=DEVICE)

        loss = (
            F.cross_entropy(logits, labels) +
            F.cross_entropy(logits.T, labels)
        ) / 2

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        pbar.set_postfix(loss=float(loss.detach()))

    # 💾 SAVE AFTER EACH EPOCH
    torch.save(model.state_dict(), CHECKPOINT_PATH)
    print(f"💾 Saved checkpoint after epoch {epoch+1}")

print("✅ Training finished safely")
torch.save(
    {
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "epoch": epoch,
    },
    CHECKPOINT_PATH
)

print("✅ Fine-tuned model saved")



✅ df loaded: (140200, 8)
✅ Vision frozen, text trainable
Training pairs: 140200
Using cached video embeddings: (7010, 512)


Epoch 1/2: 100%|██████████| 2191/2191 [13:45<00:00,  2.65it/s, loss=0.908]


💾 Saved checkpoint after epoch 1


Epoch 2/2: 100%|██████████| 2191/2191 [13:44<00:00,  2.66it/s, loss=0.986]


💾 Saved checkpoint after epoch 2
✅ Training finished safely
✅ Fine-tuned model saved


In [ ]:
#assert Path("clip_text_finetuned.pt").exists()
print("✅ File confirmed")
model.eval()
print("✅ Loaded fine-tuned CLIP text encoder")


✅ File confirmed
✅ Loaded fine-tuned CLIP text encoder


In [ ]:
# -----------------------------
# Text encoding (queries + narration)
# -----------------------------
@torch.no_grad()
def encode_texts(texts):
    toks = tokenizer(texts).to(DEVICE)
    text_emb = model.encode_text(toks)
    text_emb = F.normalize(text_emb, dim=-1)
    return text_emb


# -----------------------------
# Image encoding (frames)
# -----------------------------
@torch.no_grad()
def encode_images(pil_images):
    imgs = torch.stack([preprocess(im) for im in pil_images]).to(DEVICE)
    img_emb = model.encode_image(imgs)
    img_emb = F.normalize(img_emb, dim=-1)
    return img_emb


# -----------------------------
# Video helpers
# -----------------------------
def list_frames(video_dir):
    if not os.path.exists(video_dir):
        return []   # ✅ correct safety fix
    exts = (".jpg", ".jpeg", ".png", ".webp")
    files = [f for f in os.listdir(video_dir) if f.lower().endswith(exts)]
    files.sort()
    return [os.path.join(video_dir, f) for f in files]


def sample_frame_paths(frame_paths, num_frames):
    if len(frame_paths) == 0:
        return []
    if len(frame_paths) <= num_frames:
        return frame_paths
    idxs = np.linspace(0, len(frame_paths) - 1, num_frames).astype(int)
    return [frame_paths[i] for i in idxs]


# -----------------------------
# Video encoding (CLIP4Clip baseline)
# -----------------------------
@torch.no_grad()
def encode_video_from_frames(video_id, num_frames=12):
    vdir = os.path.join(FRAMES_ROOT, video_id)
    frame_paths = list_frames(vdir)
    frame_paths = sample_frame_paths(frame_paths, num_frames)

    if len(frame_paths) == 0:
        return None

    pil_images = [Image.open(p).convert("RGB") for p in frame_paths]

    all_emb = []
    for i in range(0, len(pil_images), BATCH_IMAGE):
        chunk = pil_images[i:i + BATCH_IMAGE]
        all_emb.append(encode_images(chunk).cpu())

    all_emb = torch.cat(all_emb, dim=0)
    vid_emb = all_emb.mean(dim=0, keepdim=True)
    vid_emb = F.normalize(vid_emb, dim=-1)

    return vid_emb.squeeze(0)


In [ ]:
# -----------------------------
# Sanity check: narrations (robust)
# -----------------------------
import json
from pathlib import Path
import random

parent = Path(FRAMES_ROOT).parent
files = list(parent.glob("video_narrations_*.json"))

assert len(files) > 0, "❌ No narration JSON files found!"
assert len(files) == 1, f"⚠️ Multiple narration files found: {[f.name for f in files]}"

NARRATION_PATH = files[0]
print("✅ Using narration file:", NARRATION_PATH.name)

with open(NARRATION_PATH, "r", encoding="utf-8") as f:
    narrations = json.load(f)


print("✅ Narration file loaded")
print("Total narrations:", len(narrations))

# --- coverage check ---
missing = [vid for vid in video_ids if vid not in narrations or narrations[vid].strip() == ""]
print("Videos missing narration:", len(missing))

# --- length sanity ---
lengths = [len(v.split()) for v in narrations.values() if v.strip()]
print("Min words:", min(lengths))
print("Max words:", max(lengths))
print("Avg words:", sum(lengths) / len(lengths))

# --- show random samples ---
print("\n🔍 Random samples:\n")
for vid in random.sample(list(narrations.keys()), 5):
    print(f"{vid}:")
    print(narrations[vid])
    print("-" * 60)


✅ Using narration file: video_narrations_blip2.json
✅ Narration file loaded
Total narrations: 7010
Videos missing narration: 0
Min words: 2
Max words: 40
Avg words: 10.014550641940085

🔍 Random samples:

video2930:
a man is standing in front of a fire
------------------------------------------------------------
video6406:
a dog is standing in the grass with its owner
------------------------------------------------------------
video4661:
a blonde woman in a pink shirt and headphones
------------------------------------------------------------
video6125:
spongebob and patrick in the car
------------------------------------------------------------
video1952:
a view of the city from the top of a building
------------------------------------------------------------


In [ ]:
# -----------------------------
# SAFE visual embedding build (resumable, save every 50)
# -----------------------------
import numpy as np, json, time
from pathlib import Path
from tqdm import tqdm
import faiss

VIDEO_EMB_PATH = Path(VEDIO_EMBEDD)                 # e.g. msrvtt_video_embs.npy
IDS_PATH       = VIDEO_EMB_PATH.with_suffix(".ids.json")
SAVE_EVERY     = 50

# ---- resume or fresh ----
if VIDEO_EMB_PATH.exists() and IDS_PATH.exists():
    mat = np.load(VIDEO_EMB_PATH)                   # [N, D]
    valid_video_ids = json.load(open(IDS_PATH, "r"))
    video_emb_list = [mat[i] for i in range(mat.shape[0])]  # list of [D]
    done = set(valid_video_ids)
    print(f"🔁 Resuming: {len(done)} videos already encoded")
else:
    video_emb_list = []
    valid_video_ids = []
    done = set()
    print("🆕 Starting fresh encoding")

t0 = time.perf_counter()
since_save = 0

for vid in tqdm(video_ids, desc="Encoding videos (visual)"):
    if vid in done:
        continue

    emb = encode_video_from_frames(vid, NUM_FRAMES)
    if emb is None:
        continue

    emb = emb.detach().cpu().numpy().astype("float32")  # [D]
    video_emb_list.append(emb)
    valid_video_ids.append(vid)
    since_save += 1

    if since_save >= SAVE_EVERY:
        np.save(VIDEO_EMB_PATH, np.stack(video_emb_list, axis=0))
        json.dump(valid_video_ids, open(IDS_PATH, "w"))
        since_save = 0

# final save
np.save(VIDEO_EMB_PATH, np.stack(video_emb_list, axis=0))
json.dump(valid_video_ids, open(IDS_PATH, "w"))

elapsed = time.perf_counter() - t0
print(f"✅ Visual encoding done in {elapsed/3600:.2f} hours")
print("Total encoded:", len(valid_video_ids))

# ---- build FAISS visual index (cosine via inner product; embeddings already normalized) ----
video_embs = np.load(VIDEO_EMB_PATH).astype("float32")   # [V, D]
faiss_index = faiss.IndexFlatIP(video_embs.shape[1])
faiss_index.add(video_embs)

vid2idx = {v:i for i,v in enumerate(valid_video_ids)}
print("✅ FAISS visual index ready:", video_embs.shape)


🔁 Resuming: 7010 videos already encoded


Encoding videos (visual): 100%|██████████| 7010/7010 [00:00<00:00, 2236068.98it/s]

✅ Visual encoding done in 0.00 hours
Total encoded: 7010
✅ FAISS visual index ready: (7010, 512)


In [ ]:
# -----------------------------
# Load MSR-VTT test captions (queries)
# -----------------------------
import pandas as pd

df_queries = pd.read_csv(CAPTIONS_CSV)
assert {"video_id", "caption"}.issubset(df_queries.columns)

print("Total queries:", len(df_queries))
print("Unique videos:", df_queries["video_id"].nunique())

df_queries.head()


Total queries: 140200
Unique videos: 7010


,video_id,caption,video_path,frames_dir,num_frames,duration,width,height
0,video0,a car is shown,/content/drive/MyDrive/MSR_VTT/raw/TrainValVid...,/content/drive/MyDrive/MSR_VTT/frames/video0,96,12.0,320,240
1,video0,a group is dancing,/content/drive/MyDrive/MSR_VTT/raw/TrainValVid...,/content/drive/MyDrive/MSR_VTT/frames/video0,96,12.0,320,240
2,video0,a man drives a vehicle through the countryside,/content/drive/MyDrive/MSR_VTT/raw/TrainValVid...,/content/drive/MyDrive/MSR_VTT/frames/video0,96,12.0,320,240
3,video0,a man drives down the road in an audi,/content/drive/MyDrive/MSR_VTT/raw/TrainValVid...,/content/drive/MyDrive/MSR_VTT/frames/video0,96,12.0,320,240
4,video0,a man driving a car,/content/drive/MyDrive/MSR_VTT/raw/TrainValVid...,/content/drive/MyDrive/MSR_VTT/frames/video0,96,12.0,320,240


In [ ]:
# -----------------------------
# Build narration embeddings (CLIP text encoder) + cache
# -----------------------------
import numpy as np, json
from pathlib import Path
from tqdm import tqdm

NARR_EMB_PATH = Path(FRAMES_ROOT).parent / f"msrvtt_narr_text_embs_{MODEL_NAME}.npy"
NARR_IDS_PATH = NARR_EMB_PATH.with_suffix(".ids.json")

if NARR_EMB_PATH.exists() and NARR_IDS_PATH.exists():
    narration_embs = np.load(NARR_EMB_PATH).astype("float32")
    narr_ids = json.load(open(NARR_IDS_PATH, "r"))
    print("✅ Loaded cached narration embeddings:", narration_embs.shape)
else:
    narr_list = []
    narr_ids = []

    batch = []
    batch_ids = []

    BATCH = 256 if DEVICE == "cuda" else 64

    for vid in tqdm(valid_video_ids, desc="Encoding narrations"):
        txt = narrations.get(vid, "").strip()
        if txt == "":
            txt = " "  # avoid empty string issues

        batch.append(txt)
        batch_ids.append(vid)

        if len(batch) >= BATCH:
            emb = encode_texts(batch).detach().cpu().numpy().astype("float32")
            narr_list.append(emb)
            narr_ids.extend(batch_ids)
            batch, batch_ids = [], []

    if len(batch) > 0:
        emb = encode_texts(batch).detach().cpu().numpy().astype("float32")
        narr_list.append(emb)
        narr_ids.extend(batch_ids)

    narration_embs = np.vstack(narr_list).astype("float32")
    np.save(NARR_EMB_PATH, narration_embs)
    json.dump(narr_ids, open(NARR_IDS_PATH, "w"))

    print("✅ Saved narration embeddings:", narration_embs.shape)

# sanity: ids must match valid_video_ids order
assert narr_ids == valid_video_ids, "❌ Narration embeddings IDs do not match visual IDs order!"


✅ Loaded cached narration embeddings: (7010, 512)


In [ ]:
# -----------------------------
# Fuse visual + narration embeddings + FAISS
# -----------------------------
import numpy as np
import faiss

alpha = 0.5  # tune later if needed

# IMPORTANT: use in-memory embeddings to guarantee alignment
video_embs = video_embs.astype("float32")        # [V, D]
# narration_embs already loaded as [V, D]

assert video_embs.shape == narration_embs.shape, "❌ Shape mismatch!"

# linear fusion
fused_video_embs = alpha * video_embs + (1 - alpha) * narration_embs

# normalize (required for cosine similarity via IP)
fused_video_embs /= np.linalg.norm(fused_video_embs, axis=1, keepdims=True) + 1e-12
fused_video_embs = fused_video_embs.astype("float32")

# FAISS index
faiss_index_fused = faiss.IndexFlatIP(fused_video_embs.shape[1])
faiss_index_fused.add(fused_video_embs)

print("✅ Fused index ready:", fused_video_embs.shape)


✅ Fused index ready: (7010, 512)


In [ ]:
# -----------------------------
# Encode text queries (with progress bar)
# -----------------------------
from tqdm import tqdm
import numpy as np

@torch.no_grad()
def encode_queries(texts, batch_size=64):
    all_embs = []
    total = len(texts)

    for i in tqdm(
        range(0, total, batch_size),
        desc="Encoding text queries in 64 batch size",
        unit="batch"
    ):
        batch = texts[i:i + batch_size]

        # CLIP text encoding (on GPU if available)
        emb = encode_texts(batch)

        # Move to CPU immediately to free GPU memory
        emb = emb.cpu().numpy().astype("float32")

        all_embs.append(emb)

    # Stack into [num_queries, embed_dim]
    return np.vstack(all_embs)

# ---- run encoding ----
query_texts = df["caption"].tolist()
query_embs = encode_queries(query_texts, batch_size=64)

print("✅ Query encoding complete")
print("Query embeddings shape:", query_embs.shape)


Encoding text queries in 64 batch size: 100%|██████████| 2191/2191 [04:39<00:00,  7.85batch/s]


✅ Query encoding complete
Query embeddings shape: (140200, 512)


# **Results**

In [ ]:
def average_precision(ranked_list, relevant_set):
    hits = 0
    sum_prec = 0.0
    for i, vid in enumerate(ranked_list, start=1):
        if vid in relevant_set:
            hits += 1
            sum_prec += hits / i
    return sum_prec / len(relevant_set) if relevant_set else 0.0

In [ ]:
import random
import numpy as np
import torch

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print("✅ Global seed fixed to", SEED)


✅ Global seed fixed to 42


In [ ]:
def _announce_mode(mode):
    print("\n" + "=" * 60)
    print(f"🔍 EVALUATION MODE: {mode.upper()}")
    if mode == "visual":
        print("• Index: faiss_index (visual-only)")
        print("• Embeddings: video_embs")
    elif mode == "visual+narration":
        print("• Index: faiss_index_fused (visual + narration)")
        print("• Embeddings: fused_video_embs")
    elif mode == "visual+rerank":
        print("• Stage 1: faiss_index (visual)")
        print("• Stage 2: cross-encoder rerank")
        print(f"• RERANK_K = {RERANK_K}, BETA = {BETA}")
    print("=" * 60)


In [ ]:
from tqdm import tqdm
import numpy as np
import time
import torch

# -----------------------------------------
# Debug controls (SAFE defaults)
# -----------------------------------------
DEBUG = False
DEBUG_EVERY = 1000

@torch.no_grad()
def retrieve_and_eval(
    df_queries,
    topk=(1, 5, 10),
    mode="visual",   # "visual", "visual+narration", "visual+rerank"
    use_faiss=True,
    max_queries=None,
    query_batch=32,
    shuffle=False,
):
    assert mode in ("visual", "visual+narration", "visual+rerank")

    _announce_mode(mode)

    # -----------------------------------------
    # Subsample queries
    # -----------------------------------------
    if max_queries is not None and len(df_queries) > max_queries:
        dfq = df_queries.sample(n=max_queries) if shuffle else df_queries.head(max_queries)
    else:
        dfq = df_queries

    captions = dfq["caption"].tolist()
    gt_vids = dfq["video_id"].tolist()

    ranks, aps, retrieval_times = [], [], []
    recall_at = {k: 0 for k in topk}

    V = len(valid_video_ids)
    if mode == "visual+rerank":
        K = min(RERANK_K, V)

    # -----------------------------------------
    # Main loop
    # -----------------------------------------
    for i in tqdm(
        range(0, len(captions), query_batch),
        desc=f"Retrieving ({mode})",
        unit="batch"
    ):
        batch_caps = captions[i:i + query_batch]
        batch_gt = gt_vids[i:i + query_batch]

        # ---- CLIP text embeddings ----
        text_embs = encode_texts(batch_caps).cpu().numpy().astype("float32")

        t0 = time.perf_counter()

        # ==================================================
        # 1️⃣ VISUAL ONLY
        # ==================================================
        if mode == "visual":
            scores, idxs = faiss_index.search(text_embs, k=V)

        # ==================================================
        # 2️⃣ VISUAL + NARRATION (fused index)
        # ==================================================
        elif mode == "visual+narration":
            scores, idxs = faiss_index_fused.search(text_embs, k=V)

        # ==================================================
        # 3️⃣ VISUAL + RERANK (QUERY-ADAPTIVE)
        # ==================================================
        else:
            # ---- Stage 1: visual retrieval ----
            vis_scores, vis_idxs = faiss_index.search(text_embs, k=V)
            topk_idxs = vis_idxs[:, :K]

            cand_vid_ids = [
                [valid_video_ids[j] for j in topk_idxs[b]]
                for b in range(len(batch_caps))
            ]

            # ---- Stage 2: reranker ----
            rerank_scores = _rerank_pairs_scores(batch_caps, cand_vid_ids)

            # ---- Query–narration similarity (CLIP space) ----
            cand_narr_embs = narration_embs[topk_idxs]  # [B,K,D]
            qn_sim = np.einsum("bd,bkd->bk", text_embs, cand_narr_embs)

            # adaptive beta per query
            beta_q = np.clip(qn_sim.mean(axis=1, keepdims=True), 0.05, 0.35)

            # ---- z-score normalize ----
            def z(x):
                return (x - x.mean(axis=1, keepdims=True)) / (
                    x.std(axis=1, keepdims=True) + 1e-6
                )

            fused = (1 - beta_q) * z(vis_scores[:, :K]) + beta_q * z(rerank_scores)
            new_order = np.argsort(-fused, axis=1)

            idxs = vis_idxs.copy()
            for b in range(len(batch_caps)):
                idxs[b, :K] = topk_idxs[b][new_order[b]]

        t_batch = time.perf_counter() - t0
        per_query = t_batch / len(batch_caps)

        # -----------------------------------------
        # Metrics
        # -----------------------------------------
        for b in range(len(batch_caps)):
            gt = batch_gt[b]
            gt_idx = vid2idx[gt]

            pos = np.where(idxs[b] == gt_idx)[0]
            gt_rank = int(pos[0]) + 1

            ranks.append(gt_rank)
            for k in topk:
                if gt_rank <= k:
                    recall_at[k] += 1

            aps.append(1.0 / gt_rank)
            retrieval_times.append(per_query)

            if DEBUG and (i + b) % DEBUG_EVERY == 0:
                print("\n[DEBUG]")
                print("Query:", batch_caps[b])
                print("GT:", gt)
                print("Top-5:", [valid_video_ids[x] for x in idxs[b][:5]])

    # -----------------------------------------
    # Aggregate
    # -----------------------------------------
    Q = len(dfq)
    recall_at = {k: recall_at[k] / Q for k in topk}

    return {
        "Queries evaluated": Q,
        "R@K": recall_at,
        "MedR": float(np.median(ranks)),
        "MeanR": float(np.mean(ranks)),
        "mAP": float(np.mean(aps)),
        "Avg retrieval time per query (s)": float(np.mean(retrieval_times)),
    }


In [ ]:
# 1️⃣ Visual
results_visual = retrieve_and_eval(
    df,
    mode="visual",
    max_queries=1000,
    shuffle=False
)

#results_visual



🔍 EVALUATION MODE: VISUAL
• Index: faiss_index (visual-only)
• Embeddings: video_embs


Retrieving (visual): 100%|██████████| 32/32 [00:03<00:00, 10.12batch/s]


In [ ]:
# 1️⃣ Visual baseline
retrieve_and_eval(df, mode="visual", max_queries=1000)


🔍 EVALUATION MODE: VISUAL
• Index: faiss_index (visual-only)
• Embeddings: video_embs


Retrieving (visual): 100%|██████████| 32/32 [00:02<00:00, 11.93batch/s]


{'Queries evaluated': 1000,
 'R@K': {1: 0.222, 5: 0.476, 10: 0.573},
 'MedR': 7.0,
 'MeanR': 100.484,
 'mAP': 0.3455360967793417,
 'Avg retrieval time per query (s)': 0.0010616739529987172}

In [ ]:
# -----------------------------
# Retrieval + eval with OPTIONAL cross-encoder rerank on TOP-K
# -----------------------------
import numpy as np
import time
from tqdm import tqdm
import torch
import torch.nn.functional as F

RERANK_K = 20         # rerank only top 50 (cheap)
GAMMA = 0.15           # how much reranker influences final order (0.1~0.3 is typical)
PAIR_BATCH = 128       # reranker pair batch size (lower if GPU memory issues)
BETA = 0.15   # 0.1–0.3 is typical; start here



def _rerank_pairs_scores(queries, cand_vid_ids):
    """
    queries: list[str] length B
    cand_vid_ids: list[list[str]] length B, each inner length K
    returns: np.ndarray [B, K] rerank scores
    """
    B = len(queries)
    K = len(cand_vid_ids[0])

    # Build flattened pairs: (query, narration(video))
    pairs_a, pairs_b = [], []
    for b in range(B):
        q = queries[b]
        for vid in cand_vid_ids[b]:
            pairs_a.append(q)
            # narration text (fallback to empty string if missing)
            pairs_b.append(narrations.get(vid, ""))

    # Run cross-encoder in batches
    scores = []
    with torch.no_grad():
        for i in range(0, len(pairs_a), PAIR_BATCH):
            a = pairs_a[i:i+PAIR_BATCH]
            b = pairs_b[i:i+PAIR_BATCH]
            inputs = rerank_tokenizer(
                a, b,
                padding=True,
                truncation=True,
                max_length=128,
                return_tensors="pt"
            ).to(DEVICE_RERANK)

            logits = rerank_model(**inputs).logits.squeeze(-1)  # [batch]
            scores.append(logits.detach().cpu())

    scores = torch.cat(scores, dim=0).view(B, K).numpy()  # [B, K]
    return scores


NARRATIONS


In [ ]:
# -----------------------------
# Encode video narrations with CLIP
# -----------------------------
@torch.no_grad()
def encode_narrations(narrations_dict, video_ids, batch_size=64):
    texts = [narrations_dict[vid] for vid in video_ids]
    all_embs = []

    for i in tqdm(range(0, len(texts), batch_size), desc="Encoding narrations"):
        batch = texts[i:i+batch_size]

        emb = encode_texts(batch).cpu().numpy().astype("float32")
        all_embs.append(emb)

    return np.vstack(all_embs)

narration_embs = encode_narrations(narrations, valid_video_ids)
print("Narration embeddings shape:", narration_embs.shape)


Encoding narrations: 100%|██████████| 110/110 [00:13<00:00,  8.32it/s]

Narration embeddings shape: (7010, 512)


In [ ]:
# -----------------------------
# Ensure embeddings are numpy arrays before fusion
# -----------------------------
import numpy as np

# If video_embs is a list (from safe/resumable build), convert it
if isinstance(video_embs, list):
    video_embs = np.stack(video_embs, axis=0).astype("float32")

# narration_embs should already be numpy, but sanity-check
if isinstance(narration_embs, list):
    narration_embs = np.stack(narration_embs, axis=0).astype("float32")

print("Visual embeddings shape:", video_embs.shape)
print("Narration embeddings shape:", narration_embs.shape)


Visual embeddings shape: (7010, 512)
Narration embeddings shape: (7010, 512)


In [ ]:
# -----------------------------
# Visual + narration fusion (mean)
# -----------------------------
alpha = 0.1   # 10% narration, 90% visual

fused_video_embs = (1 - alpha) * video_embs + alpha * narration_embs

# normalize for cosine similarity
fused_video_embs /= np.linalg.norm(fused_video_embs, axis=1, keepdims=True)


print("✅ Fused embeddings shape:", fused_video_embs.shape)


✅ Fused embeddings shape: (7010, 512)


In [ ]:
# 2️⃣ Visual + narration (fused embeddings)
#retrieve_and_eval(df, mode="visual+narration", max_queries=1000)

In [ ]:
# -----------------------------
# Load cross-encoder reranker (ONCE)
# -----------------------------
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

RERANK_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"
DEVICE_RERANK = "cuda" if torch.cuda.is_available() else "cpu"

rerank_tokenizer = AutoTokenizer.from_pretrained(RERANK_MODEL_NAME)
rerank_model = AutoModelForSequenceClassification.from_pretrained(
    RERANK_MODEL_NAME
).to(DEVICE_RERANK)

rerank_model.eval()

print("✅ Reranker loaded:", RERANK_MODEL_NAME)
print("   Device:", DEVICE_RERANK)


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Reranker loaded: cross-encoder/ms-marco-MiniLM-L-6-v2
   Device: cuda


In [ ]:
# 1️⃣ Visual baseline
results_visual=retrieve_and_eval(df, mode="visual", max_queries=1000)
# 2️⃣ Visual + narration (fused embeddings)
results_narration=retrieve_and_eval(df, mode="visual+narration", max_queries=1000)
# 3️⃣ Visual + rerank (strongest, slowest)
results_rerank=retrieve_and_eval(df, mode="visual+rerank", max_queries=1000)


🔍 EVALUATION MODE: VISUAL
• Index: faiss_index (visual-only)
• Embeddings: video_embs


Retrieving (visual): 100%|██████████| 32/32 [00:02<00:00, 11.26batch/s]



🔍 EVALUATION MODE: VISUAL+NARRATION
• Index: faiss_index_fused (visual + narration)
• Embeddings: fused_video_embs


Retrieving (visual+narration): 100%|██████████| 32/32 [00:02<00:00, 11.03batch/s]



🔍 EVALUATION MODE: VISUAL+RERANK
• Stage 1: faiss_index (visual)
• Stage 2: cross-encoder rerank
• RERANK_K = 20, BETA = 0.15


Retrieving (visual+rerank): 100%|██████████| 32/32 [00:10<00:00,  3.11batch/s]


In [ ]:
import pandas as pd

summary = pd.DataFrame.from_dict(
    {
        "Visual": results_visual,

        "Visual + Narration": results_narration,
        "Visual + Rerank": results_rerank,
    },
    orient="index"
)

summary


,Queries evaluated,R@K,MedR,MeanR,mAP,Avg retrieval time per query (s)
Visual,1000,"{1: 0.222, 5: 0.476, 10: 0.573}",7.0,100.484,0.345536,0.001110
Visual + Narration,1000,"{1: 0.175, 5: 0.422, 10: 0.517}",10.0,115.464,0.299510,0.001268
Visual + Rerank,1000,"{1: 0.206, 5: 0.483, 10: 0.567}",6.0,100.439,0.337475,0.008419
